
# 07_03D — LightGBM/XGBoost com texto — Early-Stage

**Projeto:** `dsc_legal_intelligence`  
**Target:** `target_banco_ganhou` com semântica corrigida:

- `0 = banco ganhou`
- `1 = banco perdeu`

Este notebook testa modelos compatíveis com compliance, sem `catboost`, mas agora mantendo **features textuais via TF-IDF** para uma comparação mais justa com o baseline 3A.

## Objetivo

Comparar modelos baseados em árvores/boosting usando:

1. Features numéricas early-stage + históricas sem leakage.
2. Features categóricas com encoding ordinal ou supervisionado.
3. Features textuais com `TfidfVectorizer`.
4. Avaliação walk-forward no conjunto Dev.
5. Avaliação final no Holdout temporal.
6. Métricas de negócio: PR-AUC para perda, Top-k risco de perda e Top-k financeiro.

## Por que este notebook existe?

No 07_03C, XGBoost/LightGBM foram testados sem texto e não superaram o 3A. Como o 3A usa TF-IDF, este notebook roda uma comparação mais justa adicionando texto também aos modelos 3C.



## 1. Imports e configurações

A célula abaixo detecta automaticamente se `xgboost`, `lightgbm` e `category_encoders` estão instalados. Se alguma biblioteca não existir, o respectivo modelo é ignorado.


In [ ]:

import os
import re
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    classification_report,
    confusion_matrix,
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
TARGET_COL = "target_banco_ganhou"   # 0 = banco ganhou | 1 = banco perdeu
DATE_COL = "Datainicial"
VALUE_COL_CANDIDATES = ["Valorajuizado", "valorajuizado", "ValorAjuizado", "fe_valor_ajuizado"]

# Modo rápido roda menos combinações. Use "full" para rodar mais encoders/modelos.
RUN_MODE = "fast"  # "fast" ou "full"

# Texto ligado neste notebook.
USE_TEXT_FEATURES = True

# Ajustes de TF-IDF. Se ficar lento, reduza max_features ou aumente min_df.
TFIDF_MAX_FEATURES = 1500
TFIDF_MIN_DF = 20
TFIDF_NGRAM_RANGE = (1, 2)

# Caminhos esperados gerados pelos notebooks 07_01 e 07_02
DATA_DIR = Path("../data/processed")
REPORT_DIR = Path("../outputs/reports/modelagem_early_stage")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

DEV_PATH = DATA_DIR / "abt_early_stage_v1_dev_hist.parquet"
HOLDOUT_PATH = DATA_DIR / "abt_early_stage_v1_holdout_hist.parquet"
FEATURE_LIST_PATH = DATA_DIR / "feature_list_early_stage_v1_hist.txt"

print("Configuração:")
print("RUN_MODE:", RUN_MODE)
print("USE_TEXT_FEATURES:", USE_TEXT_FEATURES)
print("DEV_PATH:", DEV_PATH)
print("HOLDOUT_PATH:", HOLDOUT_PATH)
print("FEATURE_LIST_PATH:", FEATURE_LIST_PATH)


In [ ]:

# Detecção de bibliotecas opcionais
HAS_XGBOOST = False
HAS_LIGHTGBM = False
HAS_CATEGORY_ENCODERS = False

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception as e:
    print("XGBoost indisponível:", repr(e))

try:
    from lightgbm import LGBMClassifier
    HAS_LIGHTGBM = True
except Exception as e:
    print("LightGBM indisponível:", repr(e))

try:
    import category_encoders as ce
    HAS_CATEGORY_ENCODERS = True
except Exception as e:
    print("category_encoders indisponível:", repr(e))

print({
    "HAS_XGBOOST": HAS_XGBOOST,
    "HAS_LIGHTGBM": HAS_LIGHTGBM,
    "HAS_CATEGORY_ENCODERS": HAS_CATEGORY_ENCODERS,
})



## 2. Funções auxiliares


In [ ]:

def save_report(df: pd.DataFrame, filename: str) -> Path:
    path = REPORT_DIR / filename
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"Relatório salvo em: {path}")
    return path


def read_feature_list(path: Path) -> list[str]:
    if not path.exists():
        raise FileNotFoundError(f"Feature list não encontrada: {path}")
    with open(path, "r", encoding="utf-8") as f:
        feats = [line.strip() for line in f if line.strip()]
    return feats


def get_value_col(df: pd.DataFrame):
    for c in VALUE_COL_CANDIDATES:
        if c in df.columns:
            return c
    return None


def safe_text_concat(df: pd.DataFrame, text_cols: list[str]) -> pd.Series:
    if not text_cols:
        return pd.Series([""] * len(df), index=df.index)
    return (
        df[text_cols]
        .fillna("")
        .astype(str)
        .agg(" ".join, axis=1)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


def best_f1_threshold(y_true, proba_perda):
    precision, recall, thresholds = precision_recall_curve(y_true, proba_perda)
    # thresholds tem tamanho n-1; precision/recall têm n
    f1 = 2 * precision * recall / np.clip(precision + recall, 1e-12, None)
    if len(thresholds) == 0:
        return 0.5, precision[0], recall[0], f1[0]
    best_idx = int(np.nanargmax(f1[:-1]))
    return float(thresholds[best_idx]), float(precision[best_idx]), float(recall[best_idx]), float(f1[best_idx])


def metrics_at_threshold(y_true, proba_perda, threshold=0.5):
    pred = (proba_perda >= threshold).astype(int)
    tp = ((pred == 1) & (y_true == 1)).sum()
    fp = ((pred == 1) & (y_true == 0)).sum()
    fn = ((pred == 0) & (y_true == 1)).sum()
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    return {
        "threshold": threshold,
        "precision_perda": precision,
        "recall_perda": recall,
        "f1_perda": f1,
        "pred_perda_rate": pred.mean(),
    }


def evaluate_proba(y_true, proba_perda, prefix=""):
    y_true = np.asarray(y_true).astype(int)
    proba_perda = np.asarray(proba_perda)
    roc = roc_auc_score(y_true, proba_perda)
    pr = average_precision_score(y_true, proba_perda)
    best_thr, best_p, best_r, best_f1 = best_f1_threshold(y_true, proba_perda)
    thr_05 = metrics_at_threshold(y_true, proba_perda, threshold=0.5)
    out = {
        f"{prefix}roc_auc_perda": roc,
        f"{prefix}pr_auc_perda": pr,
        f"{prefix}taxa_perda": y_true.mean(),
        f"{prefix}taxa_ganho": 1 - y_true.mean(),
        f"{prefix}best_f1_threshold_perda": best_thr,
        f"{prefix}best_f1_precision_perda": best_p,
        f"{prefix}best_f1_recall_perda": best_r,
        f"{prefix}best_f1_perda": best_f1,
        f"{prefix}precision_perda_t05": thr_05["precision_perda"],
        f"{prefix}recall_perda_t05": thr_05["recall_perda"],
        f"{prefix}f1_perda_t05": thr_05["f1_perda"],
        f"{prefix}pred_perda_rate_t05": thr_05["pred_perda_rate"],
    }
    return out


def get_proba_perda(model, X):
    """Retorna P(classe 1 = banco perdeu)."""
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X)
        # Classe positiva é 1. Garantir índice correto.
        if hasattr(model, "classes_"):
            classes = list(model.classes_)
            if 1 in classes:
                idx = classes.index(1)
            else:
                idx = 1
        else:
            idx = 1
        return proba[:, idx]
    if hasattr(model, "decision_function"):
        score = model.decision_function(X)
        # transformação logística simples
        return 1 / (1 + np.exp(-score))
    raise ValueError("Modelo não possui predict_proba nem decision_function.")


def topk_risco_perda_metrics(y_true, proba_perda, ks=(0.01, 0.05, 0.10, 0.20)):
    df = pd.DataFrame({"y_true": np.asarray(y_true).astype(int), "proba_perda": np.asarray(proba_perda)})
    df = df.sort_values("proba_perda", ascending=False).reset_index(drop=True)
    base = df["y_true"].mean()
    rows = []
    for k in ks:
        n = max(1, int(np.ceil(len(df) * k)))
        top = df.head(n)
        precision = top["y_true"].mean()
        recall = top["y_true"].sum() / max(df["y_true"].sum(), 1)
        rows.append({
            "top_k": k,
            "n_top": n,
            "precision_perda_at_k": precision,
            "recall_perda_at_k": recall,
            "lift_perda_at_k": precision / max(base, 1e-12),
            "taxa_perda_base": base,
        })
    return pd.DataFrame(rows)


def topk_prioridade_financeira_metrics(y_true, proba_perda, valor_ajuizado, ks=(0.01, 0.05, 0.10, 0.20)):
    df = pd.DataFrame({
        "y_true": np.asarray(y_true).astype(int),
        "proba_perda": np.asarray(proba_perda),
        "valor_ajuizado": pd.to_numeric(valor_ajuizado, errors="coerce").fillna(0).clip(lower=0),
    })
    # score financeiro simples = probabilidade de perda * exposição ajuizada
    df["score_financeiro_perda"] = df["proba_perda"] * df["valor_ajuizado"]
    df = df.sort_values("score_financeiro_perda", ascending=False).reset_index(drop=True)
    base = df["y_true"].mean()
    total_valor = df["valor_ajuizado"].sum()
    total_valor_perdas = df.loc[df["y_true"] == 1, "valor_ajuizado"].sum()
    rows = []
    for k in ks:
        n = max(1, int(np.ceil(len(df) * k)))
        top = df.head(n)
        valor_top = top["valor_ajuizado"].sum()
        valor_perdas_top = top.loc[top["y_true"] == 1, "valor_ajuizado"].sum()
        precision = top["y_true"].mean()
        recall = top["y_true"].sum() / max(df["y_true"].sum(), 1)
        rows.append({
            "top_k": k,
            "n_top": n,
            "precision_perda_at_k": precision,
            "recall_perda_at_k": recall,
            "lift_perda_at_k": precision / max(base, 1e-12),
            "taxa_perda_base": base,
            "valor_ajuizado_top": valor_top,
            "share_valor_ajuizado_total": valor_top / max(total_valor, 1e-12),
            "valor_ajuizado_perdas_top": valor_perdas_top,
            "share_valor_perdas_total": valor_perdas_top / max(total_valor_perdas, 1e-12),
        })
    return pd.DataFrame(rows)



## 3. Carregar ABTs geradas nas etapas anteriores

Este notebook espera os arquivos gerados por:

- `07_01_prepare_early_stage_abt.ipynb`
- `07_02_historical_features_no_leakage.ipynb`


In [ ]:

df_dev = pd.read_parquet(DEV_PATH)
df_holdout = pd.read_parquet(HOLDOUT_PATH)
feature_list = read_feature_list(FEATURE_LIST_PATH)

# Garantir data
for df in [df_dev, df_holdout]:
    if DATE_COL in df.columns:
        df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")

# Validar target
assert TARGET_COL in df_dev.columns, f"{TARGET_COL} não encontrado no Dev"
assert TARGET_COL in df_holdout.columns, f"{TARGET_COL} não encontrado no Holdout"

# Manter apenas features existentes
feature_list = [c for c in feature_list if c in df_dev.columns and c in df_holdout.columns]

print("df_dev:", df_dev.shape)
print("df_holdout:", df_holdout.shape)
print("features na lista:", len(feature_list))
print("target Dev:")
print(df_dev[TARGET_COL].value_counts(normalize=True).sort_index())
print("target Holdout:")
print(df_holdout[TARGET_COL].value_counts(normalize=True).sort_index())



## 4. Selecionar features numéricas, categóricas e textuais

Regra geral:

- Numéricas: `number`, `bool`, flags e históricos.
- Categóricas: objetos/string de baixa ou média cardinalidade.
- Texto: colunas textuais informativas, concatenadas em um único campo `__texto_modelo__`.

Se `USE_TEXT_FEATURES = True`, o texto entra no pipeline via TF-IDF.


In [ ]:

# Candidatas textuais. Ajuste conforme os nomes existentes no seu dataframe.
TEXT_CANDIDATES = [
    "Nm_Assunto", "Nm_Acao", "Nm_Produto", "Carteira", "Comarca", "UF",
    "assunto_texto", "assunto_normalizado_texto", "classe_texto", "area_texto",
    "tipo_de_justica_texto", "tipo_de_processo_texto", "fase_processual_texto",
    "orgao_julgador_texto", "vara_texto", "cidade", "uf",
    "fe_nm_assunto_norm", "fe_nm_acao_norm", "fe_nm_produto_norm", "fe_uf_norm", "fe_comarca_norm",
]

# Bloqueios explícitos por segurança. Não usar campos pós-desfecho/leakage.
LEAKAGE_PATTERNS = [
    r"motivo_encerramento", r"dataencerramento", r"situacao$", r"desfecho", r"resultado",
    r"sentenca", r"conden", r"acordo_prob", r"procedente", r"transito", r"valor_arbitrado",
    r"valor_do_acordo", r"provavel", r"prob$", r"target",
]

def is_leakage_name(col):
    c = str(col).lower()
    return any(re.search(p, c) for p in LEAKAGE_PATTERNS)

feature_list_safe = [c for c in feature_list if not is_leakage_name(c)]
removed_by_name = sorted(set(feature_list) - set(feature_list_safe))
print("Features removidas por possível leakage pelo nome:", len(removed_by_name))
if removed_by_name[:30]:
    print(removed_by_name[:30])

text_cols = [c for c in TEXT_CANDIDATES if c in df_dev.columns and c in df_holdout.columns and not is_leakage_name(c)]
if not USE_TEXT_FEATURES:
    text_cols = []

# Evitar duplicar colunas textuais também como categóricas se elas entrarem no campo concatenado.
base_features = [c for c in feature_list_safe if c not in text_cols]

numeric_features = [
    c for c in base_features
    if pd.api.types.is_numeric_dtype(df_dev[c]) or pd.api.types.is_bool_dtype(df_dev[c])
]

categorical_features = [
    c for c in base_features
    if c not in numeric_features and c != DATE_COL
]

print("numeric_features:", len(numeric_features))
print("categorical_features:", len(categorical_features))
print("text_cols:", len(text_cols), text_cols)
print("total input bruto:", len(numeric_features) + len(categorical_features) + len(text_cols))


In [ ]:

# Criar cópias com coluna textual concatenada
TEXT_MODEL_COL = "__texto_modelo__"

df_dev_model = df_dev.copy()
df_holdout_model = df_holdout.copy()

if USE_TEXT_FEATURES and text_cols:
    df_dev_model[TEXT_MODEL_COL] = safe_text_concat(df_dev_model, text_cols)
    df_holdout_model[TEXT_MODEL_COL] = safe_text_concat(df_holdout_model, text_cols)
    text_features = [TEXT_MODEL_COL]
else:
    df_dev_model[TEXT_MODEL_COL] = ""
    df_holdout_model[TEXT_MODEL_COL] = ""
    text_features = []

selected_features = numeric_features + categorical_features + text_features

feature_inventory = pd.DataFrame({
    "feature": selected_features,
    "tipo": [
        "numeric" if c in numeric_features else "categorical" if c in categorical_features else "text_concat"
        for c in selected_features
    ],
    "missing_rate_dev": [df_dev_model[c].isna().mean() if c in df_dev_model.columns else 0 for c in selected_features],
    "missing_rate_holdout": [df_holdout_model[c].isna().mean() if c in df_holdout_model.columns else 0 for c in selected_features],
    "nunique_dev": [df_dev_model[c].nunique(dropna=True) if c in df_dev_model.columns else np.nan for c in selected_features],
    "nunique_holdout": [df_holdout_model[c].nunique(dropna=True) if c in df_holdout_model.columns else np.nan for c in selected_features],
})

save_report(feature_inventory, "70_3d_feature_inventory.csv")
feature_inventory.head(30)



## 5. Folds temporais walk-forward no Dev

Os folds são criados dentro do Dev para evitar usar o Holdout no tuning.


In [ ]:

def make_temporal_folds(df, date_col=DATE_COL, n_folds=3, min_train_frac=0.55):
    df_sorted = df.sort_values(date_col).reset_index(drop=True)
    n = len(df_sorted)
    # Cortes aproximados: treino crescente e validações sequenciais
    cuts = [0.55, 0.70, 0.85, 1.00]
    folds = []
    for i in range(n_folds):
        train_end = int(n * cuts[i])
        val_start = train_end
        val_end = int(n * cuts[i + 1])
        train_idx = df_sorted.index[:train_end].to_numpy()
        val_idx = df_sorted.index[val_start:val_end].to_numpy()
        folds.append((train_idx, val_idx, df_sorted))
    return folds

folds = make_temporal_folds(df_dev_model, DATE_COL, n_folds=3)

fold_summary_rows = []
for i, (train_idx, val_idx, df_sorted) in enumerate(folds, start=1):
    tr = df_sorted.iloc[train_idx]
    va = df_sorted.iloc[val_idx]
    fold_summary_rows.append({
        "fold": i,
        "train_start_date": tr[DATE_COL].min(),
        "train_end_date": tr[DATE_COL].max(),
        "val_start_date": va[DATE_COL].min(),
        "val_end_date": va[DATE_COL].max(),
        "qtd_train": len(tr),
        "qtd_val": len(va),
        "taxa_perda_train": tr[TARGET_COL].mean(),
        "taxa_perda_val": va[TARGET_COL].mean(),
        "taxa_ganho_train": 1 - tr[TARGET_COL].mean(),
        "taxa_ganho_val": 1 - va[TARGET_COL].mean(),
    })

fold_summary = pd.DataFrame(fold_summary_rows)
save_report(fold_summary, "71_3d_walk_forward_folds_summary.csv")
fold_summary



## 6. Pré-processadores

Serão testadas duas famílias principais:

1. `ordinal`: numéricas + categóricas com `OrdinalEncoder` + texto TF-IDF.
2. `ce_catboost`: numéricas + categóricas com `CatBoostEncoder` do `category_encoders` + texto TF-IDF.

Observação: `CatBoostEncoder` aqui é **apenas encoder**, não é o modelo CatBoost.


In [ ]:

def make_preprocessor(preprocess_mode="ordinal"):
    transformers = []

    if numeric_features:
        num_pipe = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
        ])
        transformers.append(("num", num_pipe, numeric_features))

    if categorical_features:
        if preprocess_mode == "ordinal":
            cat_pipe = Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
                ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
            ])
        elif preprocess_mode == "onehot":
            cat_pipe = Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
                ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20, sparse_output=True)),
            ])
        elif preprocess_mode == "ce_catboost":
            if not HAS_CATEGORY_ENCODERS:
                raise RuntimeError("category_encoders não instalado.")
            cat_pipe = Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
                ("catboost_encoder", ce.CatBoostEncoder(
                    cols=list(range(len(categorical_features))),
                    random_state=RANDOM_STATE,
                    sigma=None,
                )),
            ])
        elif preprocess_mode == "ce_mestimate":
            if not HAS_CATEGORY_ENCODERS:
                raise RuntimeError("category_encoders não instalado.")
            cat_pipe = Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
                ("mestimate_encoder", ce.MEstimateEncoder(
                    cols=list(range(len(categorical_features))),
                    m=10.0,
                )),
            ])
        elif preprocess_mode == "ce_jamesstein":
            if not HAS_CATEGORY_ENCODERS:
                raise RuntimeError("category_encoders não instalado.")
            cat_pipe = Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
                ("jamesstein_encoder", ce.JamesSteinEncoder(
                    cols=list(range(len(categorical_features))),
                )),
            ])
        else:
            raise ValueError(f"preprocess_mode inválido: {preprocess_mode}")

        transformers.append(("cat", cat_pipe, categorical_features))

    if text_features:
        text_pipe = Pipeline(steps=[
            ("selector", FunctionTransformer(lambda x: x.squeeze() if hasattr(x, "squeeze") else x, validate=False)),
            ("tfidf", TfidfVectorizer(
                max_features=TFIDF_MAX_FEATURES,
                min_df=TFIDF_MIN_DF,
                ngram_range=TFIDF_NGRAM_RANGE,
                strip_accents="unicode",
                lowercase=True,
            )),
        ])
        transformers.append(("txt", text_pipe, TEXT_MODEL_COL))

    return ColumnTransformer(transformers=transformers, remainder="drop", sparse_threshold=0.3)



## 7. Modelos candidatos

A célula abaixo cria candidatos conforme as bibliotecas disponíveis.


In [ ]:

def make_model_candidates():
    candidates = []

    # Baseline forte comparável ao 3A, mas aqui dentro do notebook 3D
    candidates.append((
        "rf_onehot_tfidf",
        "onehot",
        RandomForestClassifier(
            n_estimators=400,
            max_depth=18,
            min_samples_split=30,
            min_samples_leaf=15,
            max_features="sqrt",
            max_samples=0.8,
            class_weight="balanced",
            criterion="entropy",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )
    ))

    candidates.append((
        "logistic_onehot_tfidf",
        "onehot",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            solver="saga",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )
    ))

    candidates.append((
        "hgb_ordinal_tfidf",
        "ordinal",
        HistGradientBoostingClassifier(
            max_iter=250,
            learning_rate=0.05,
            max_leaf_nodes=31,
            l2_regularization=1.0,
            random_state=RANDOM_STATE,
        )
    ))

    if HAS_XGBOOST:
        xgb_params = dict(
            n_estimators=500,
            max_depth=4,
            learning_rate=0.04,
            subsample=0.85,
            colsample_bytree=0.85,
            min_child_weight=20,
            reg_lambda=5.0,
            objective="binary:logistic",
            eval_metric="aucpr",
            tree_method="hist",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
        candidates.append(("xgb_ordinal_tfidf", "ordinal", XGBClassifier(**xgb_params)))
        if HAS_CATEGORY_ENCODERS:
            candidates.append(("xgb_catboost_encoder_tfidf", "ce_catboost", XGBClassifier(**xgb_params)))
            if RUN_MODE == "full":
                candidates.append(("xgb_mestimate_encoder_tfidf", "ce_mestimate", XGBClassifier(**xgb_params)))
                candidates.append(("xgb_jamesstein_encoder_tfidf", "ce_jamesstein", XGBClassifier(**xgb_params)))

    if HAS_LIGHTGBM:
        lgbm_params = dict(
            n_estimators=600,
            learning_rate=0.04,
            num_leaves=31,
            max_depth=-1,
            min_child_samples=50,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=5.0,
            class_weight="balanced",
            objective="binary",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbose=-1,
        )
        candidates.append(("lgbm_ordinal_tfidf", "ordinal", LGBMClassifier(**lgbm_params)))
        if HAS_CATEGORY_ENCODERS:
            candidates.append(("lgbm_catboost_encoder_tfidf", "ce_catboost", LGBMClassifier(**lgbm_params)))
            if RUN_MODE == "full":
                candidates.append(("lgbm_mestimate_encoder_tfidf", "ce_mestimate", LGBMClassifier(**lgbm_params)))
                candidates.append(("lgbm_jamesstein_encoder_tfidf", "ce_jamesstein", LGBMClassifier(**lgbm_params)))

    return candidates

candidates = make_model_candidates()
print("Modelos candidatos:")
for name, prep, model in candidates:
    print("-", name, "|", prep, "|", type(model).__name__)



## 8. Walk-forward no Dev

A métrica principal é `PR-AUC` para a classe 1 = banco perdeu.


In [ ]:

wf_rows = []
topk_rows = []

for model_name, preprocess_mode, estimator in candidates:
    print(f"\nTreinando candidato: {model_name} | preprocess={preprocess_mode}")
    for fold_id, (train_idx, val_idx, df_sorted) in enumerate(folds, start=1):
        train = df_sorted.iloc[train_idx].copy()
        val = df_sorted.iloc[val_idx].copy()

        X_train = train[selected_features]
        y_train = train[TARGET_COL].astype(int)
        X_val = val[selected_features]
        y_val = val[TARGET_COL].astype(int)

        pipe = Pipeline(steps=[
            ("preprocessor", make_preprocessor(preprocess_mode)),
            ("model", clone(estimator)),
        ])

        pipe.fit(X_train, y_train)
        proba_val = get_proba_perda(pipe, X_val)
        m = evaluate_proba(y_val, proba_val, prefix="")
        m.update({
            "model": model_name,
            "preprocess_mode": preprocess_mode,
            "fold": fold_id,
            "qtd_train": len(train),
            "qtd_val": len(val),
            "val_start_date": val[DATE_COL].min(),
            "val_end_date": val[DATE_COL].max(),
        })
        wf_rows.append(m)

        tk = topk_risco_perda_metrics(y_val, proba_val)
        tk["model"] = model_name
        tk["preprocess_mode"] = preprocess_mode
        tk["fold"] = fold_id
        topk_rows.append(tk)

wf_results = pd.DataFrame(wf_rows)
topk_wf = pd.concat(topk_rows, ignore_index=True) if topk_rows else pd.DataFrame()

save_report(wf_results, "72_3d_text_walk_forward_results.csv")
save_report(topk_wf, "73_3d_text_walk_forward_topk_perda.csv")

wf_results.sort_values(["pr_auc_perda"], ascending=False).head(20)


In [ ]:

model_summary = (
    wf_results
    .groupby(["model", "preprocess_mode"], as_index=False)
    .agg(
        mean_roc_auc_perda=("roc_auc_perda", "mean"),
        std_roc_auc_perda=("roc_auc_perda", "std"),
        mean_pr_auc_perda=("pr_auc_perda", "mean"),
        std_pr_auc_perda=("pr_auc_perda", "std"),
        mean_taxa_perda_val=("taxa_perda", "mean"),
        mean_taxa_ganho_val=("taxa_ganho", "mean"),
        mean_best_f1_perda=("best_f1_perda", "mean"),
        mean_best_f1_threshold_perda=("best_f1_threshold_perda", "mean"),
        mean_precision_perda_t05=("precision_perda_t05", "mean"),
        mean_recall_perda_t05=("recall_perda_t05", "mean"),
        mean_f1_perda_t05=("f1_perda_t05", "mean"),
    )
    .sort_values("mean_pr_auc_perda", ascending=False)
)

save_report(model_summary, "74_3d_text_model_summary.csv")
model_summary



## 9. Top-k de risco de perda por modelo no walk-forward


In [ ]:

topk_summary = (
    topk_wf
    .groupby(["model", "preprocess_mode", "top_k"], as_index=False)
    .agg(
        mean_precision_perda_at_k=("precision_perda_at_k", "mean"),
        mean_recall_perda_at_k=("recall_perda_at_k", "mean"),
        mean_lift_perda_at_k=("lift_perda_at_k", "mean"),
        mean_taxa_perda_base=("taxa_perda_base", "mean"),
    )
    .sort_values(["top_k", "mean_precision_perda_at_k"], ascending=[True, False])
)

save_report(topk_summary, "75_3d_text_walk_forward_topk_summary.csv")
topk_summary



## 10. Treinar melhor modelo no Dev completo e avaliar no Holdout temporal


In [ ]:

best_row = model_summary.iloc[0]
best_model_name = best_row["model"]
best_preprocess_mode = best_row["preprocess_mode"]

best_tuple = [c for c in candidates if c[0] == best_model_name and c[1] == best_preprocess_mode][0]
_, _, best_estimator = best_tuple

print("Melhor modelo no walk-forward:", best_model_name)
print("Preprocessamento:", best_preprocess_mode)
print("Mean PR-AUC perda WF:", best_row["mean_pr_auc_perda"])

X_dev = df_dev_model[selected_features]
y_dev = df_dev_model[TARGET_COL].astype(int)
X_holdout = df_holdout_model[selected_features]
y_holdout = df_holdout_model[TARGET_COL].astype(int)

best_pipe = Pipeline(steps=[
    ("preprocessor", make_preprocessor(best_preprocess_mode)),
    ("model", clone(best_estimator)),
])

best_pipe.fit(X_dev, y_dev)
proba_perda_holdout = get_proba_perda(best_pipe, X_holdout)

holdout_m = evaluate_proba(y_holdout, proba_perda_holdout, prefix="")
holdout_metrics = pd.DataFrame([{
    "model": best_model_name,
    "preprocess_mode": best_preprocess_mode,
    "roc_auc_perda": holdout_m["roc_auc_perda"],
    "pr_auc_perda": holdout_m["pr_auc_perda"],
    "taxa_perda_holdout": holdout_m["taxa_perda"],
    "taxa_ganho_holdout": holdout_m["taxa_ganho"],
    "best_f1_threshold_perda": holdout_m["best_f1_threshold_perda"],
    "best_f1_precision_perda": holdout_m["best_f1_precision_perda"],
    "best_f1_recall_perda": holdout_m["best_f1_recall_perda"],
    "best_f1_perda": holdout_m["best_f1_perda"],
    "precision_perda_t05": holdout_m["precision_perda_t05"],
    "recall_perda_t05": holdout_m["recall_perda_t05"],
    "f1_perda_t05": holdout_m["f1_perda_t05"],
    "pred_perda_rate_t05": holdout_m["pred_perda_rate_t05"],
    "qtd_dev": len(df_dev_model),
    "qtd_holdout": len(df_holdout_model),
    "qtd_features": len(selected_features),
    "use_text_features": USE_TEXT_FEATURES,
}])

save_report(holdout_metrics, "76_3d_text_holdout_best_model_metrics.csv")
holdout_metrics



## 11. Classification report e matriz de confusão no Holdout

A classe positiva é `1 = banco perdeu`.


In [ ]:

threshold_05 = 0.5
pred_05 = (proba_perda_holdout >= threshold_05).astype(int)

print("Classification report — threshold 0.5")
print("Classe 0 = banco ganhou")
print("Classe 1 = banco perdeu")
print(classification_report(
    y_holdout,
    pred_05,
    target_names=["banco_ganhou", "banco_perdeu"],
    digits=4,
))

cm = confusion_matrix(y_holdout, pred_05)
print("Confusion matrix — threshold 0.5")
print("Linhas = real | Colunas = previsto")
print(cm)

cm_df = pd.DataFrame(
    cm,
    index=["real_banco_ganhou", "real_banco_perdeu"],
    columns=["pred_banco_ganhou", "pred_banco_perdeu"],
)
save_report(cm_df.reset_index().rename(columns={"index": "classe_real"}), "77_3d_text_holdout_confusion_matrix_t05.csv")
cm_df



## 12. Top-k no Holdout — risco puro de perda


In [ ]:

holdout_topk_perda = topk_risco_perda_metrics(
    y_true=y_holdout.values,
    proba_perda=proba_perda_holdout,
    ks=(0.01, 0.05, 0.10, 0.20),
)
holdout_topk_perda["model"] = best_model_name
holdout_topk_perda["preprocess_mode"] = best_preprocess_mode

save_report(holdout_topk_perda, "78_3d_text_holdout_topk_risco_perda.csv")
holdout_topk_perda



## 13. Top-k no Holdout — prioridade financeira

Ordena por `probabilidade de perda × Valorajuizado`.


In [ ]:

VALUE_COL = get_value_col(df_holdout_model)
print("VALUE_COL:", VALUE_COL)

if VALUE_COL is not None:
    holdout_topk_financeiro = topk_prioridade_financeira_metrics(
        y_true=y_holdout.values,
        proba_perda=proba_perda_holdout,
        valor_ajuizado=df_holdout_model[VALUE_COL],
        ks=(0.01, 0.05, 0.10, 0.20),
    )
    holdout_topk_financeiro["model"] = best_model_name
    holdout_topk_financeiro["preprocess_mode"] = best_preprocess_mode
    save_report(holdout_topk_financeiro, "79_3d_text_holdout_topk_prioridade_financeira.csv")
    display(holdout_topk_financeiro)
else:
    holdout_topk_financeiro = pd.DataFrame()
    print("Nenhuma coluna de valor ajuizado encontrada. Pulando análise financeira.")



## 14. Ranking do Holdout para consumo jurídico

Gera um arquivo com score de perda, ranking de risco e ranking financeiro.


In [ ]:

ranking_cols = []
for c in ["Numerodistribuicao", "Identificador", DATE_COL, VALUE_COL, "Nm_Assunto", "Nm_Acao", "Nm_Produto", "Carteira", "UF", "Comarca"]:
    if c and c in df_holdout_model.columns and c not in ranking_cols:
        ranking_cols.append(c)

ranking_holdout = df_holdout_model[ranking_cols].copy() if ranking_cols else pd.DataFrame(index=df_holdout_model.index)
ranking_holdout[TARGET_COL] = y_holdout.values
ranking_holdout["proba_perda"] = proba_perda_holdout
ranking_holdout["proba_ganho"] = 1 - proba_perda_holdout
ranking_holdout["rank_risco_perda"] = ranking_holdout["proba_perda"].rank(ascending=False, method="first").astype(int)

if VALUE_COL is not None:
    ranking_holdout["score_financeiro_perda"] = ranking_holdout["proba_perda"] * pd.to_numeric(df_holdout_model[VALUE_COL], errors="coerce").fillna(0).clip(lower=0)
    ranking_holdout["rank_prioridade_financeira"] = ranking_holdout["score_financeiro_perda"].rank(ascending=False, method="first").astype(int)
else:
    ranking_holdout["score_financeiro_perda"] = np.nan
    ranking_holdout["rank_prioridade_financeira"] = np.nan

ranking_path = DATA_DIR / "ranking_holdout_risco_perda_early_stage_3d_text.parquet"
ranking_holdout.to_parquet(ranking_path, index=False)
print("Ranking salvo em:", ranking_path)
ranking_holdout.head(20)



## 15. Comparação com 3A, 3B e 3C

Preencha os valores de referência com os resultados já observados nos notebooks anteriores.


In [ ]:

# Valores de referência informados nos notebooks anteriores.
# Ajuste manualmente se seus CSVs oficiais tiverem números diferentes.
comparison_rows = [
    {
        "model": "3A_random_forest_onehot_tfidf",
        "holdout_pr_auc_perda": 0.466804,
        "holdout_roc_auc_perda": 0.778580,
        "top5_precision_perda": 0.603004,
        "top10_fin_share_valor_perdas": 0.502560,
    },
    {
        "model": "3B_hgb_catboost_encoder",
        "holdout_pr_auc_perda": 0.434753,
        "holdout_roc_auc_perda": 0.769694,
        "top5_precision_perda": 0.525751,
        "top10_fin_share_valor_perdas": 0.509398,
    },
    {
        "model": "3C_xgb_ordinal_sem_texto",
        "holdout_pr_auc_perda": 0.434474,
        "holdout_roc_auc_perda": 0.764934,
        "top5_precision_perda": 0.534335,
        "top10_fin_share_valor_perdas": 0.499359,
    },
]

# Adicionar resultado 3D
row_3d = {
    "model": f"3D_{best_model_name}",
    "holdout_pr_auc_perda": float(holdout_metrics.loc[0, "pr_auc_perda"]),
    "holdout_roc_auc_perda": float(holdout_metrics.loc[0, "roc_auc_perda"]),
    "top5_precision_perda": float(holdout_topk_perda.loc[holdout_topk_perda["top_k"] == 0.05, "precision_perda_at_k"].iloc[0]),
    "top10_fin_share_valor_perdas": (
        float(holdout_topk_financeiro.loc[holdout_topk_financeiro["top_k"] == 0.10, "share_valor_perdas_total"].iloc[0])
        if not holdout_topk_financeiro.empty else np.nan
    ),
}
comparison_rows.append(row_3d)

comparison_3a_3b_3c_3d = pd.DataFrame(comparison_rows)
base = comparison_3a_3b_3c_3d.loc[comparison_3a_3b_3c_3d["model"] == "3A_random_forest_onehot_tfidf"].iloc[0]
comparison_3a_3b_3c_3d["delta_pr_auc_vs_3a"] = comparison_3a_3b_3c_3d["holdout_pr_auc_perda"] - base["holdout_pr_auc_perda"]
comparison_3a_3b_3c_3d["delta_roc_auc_vs_3a"] = comparison_3a_3b_3c_3d["holdout_roc_auc_perda"] - base["holdout_roc_auc_perda"]
comparison_3a_3b_3c_3d["delta_top5_precision_vs_3a"] = comparison_3a_3b_3c_3d["top5_precision_perda"] - base["top5_precision_perda"]
comparison_3a_3b_3c_3d["delta_top10_fin_vs_3a"] = comparison_3a_3b_3c_3d["top10_fin_share_valor_perdas"] - base["top10_fin_share_valor_perdas"]

save_report(comparison_3a_3b_3c_3d, "80_3d_text_comparison_3a_3b_3c_3d.csv")
comparison_3a_3b_3c_3d



## 16. Feature importance, se disponível

Para modelos de árvore, tenta extrair importância do estimador final. Nem sempre os nomes saem limpos depois do `ColumnTransformer`, mas já ajuda a inspecionar sinais dominantes.


In [ ]:

feature_importance = pd.DataFrame()
try:
    pre = best_pipe.named_steps["preprocessor"]
    model = best_pipe.named_steps["model"]
    feature_names = pre.get_feature_names_out()
    if hasattr(model, "feature_importances_"):
        importances = model.feature_importances_
        feature_importance = pd.DataFrame({"feature": feature_names, "importance": importances})
        feature_importance = feature_importance.sort_values("importance", ascending=False)
        save_report(feature_importance, "81_3d_text_feature_importance.csv")
        display(feature_importance.head(50))
    elif hasattr(model, "coef_"):
        coefs = model.coef_.ravel()
        feature_importance = pd.DataFrame({"feature": feature_names, "coef": coefs, "abs_coef": np.abs(coefs)})
        feature_importance = feature_importance.sort_values("abs_coef", ascending=False)
        save_report(feature_importance, "81_3d_text_feature_importance.csv")
        display(feature_importance.head(50))
    else:
        print("Modelo não possui feature_importances_ nem coef_.")
except Exception as e:
    print("Não foi possível extrair feature importance:", repr(e))



## 17. Summary final

Traga este summary para a próxima iteração.


In [ ]:

summary_step_3d = pd.DataFrame([{
    "target_semantica": "0=banco_ganhou | 1=banco_perdeu",
    "run_mode": RUN_MODE,
    "use_text_features": USE_TEXT_FEATURES,
    "tfidf_max_features": TFIDF_MAX_FEATURES,
    "tfidf_min_df": TFIDF_MIN_DF,
    "has_lightgbm": HAS_LIGHTGBM,
    "has_xgboost": HAS_XGBOOST,
    "has_category_encoders": HAS_CATEGORY_ENCODERS,
    "best_model_3d_walk_forward": best_model_name,
    "best_model_3d_preprocess": best_preprocess_mode,
    "best_model_3d_mean_pr_auc_perda_wf": float(best_row["mean_pr_auc_perda"]),
    "best_model_3d_mean_roc_auc_perda_wf": float(best_row["mean_roc_auc_perda"]),
    "holdout_pr_auc_perda": float(holdout_metrics.loc[0, "pr_auc_perda"]),
    "holdout_roc_auc_perda": float(holdout_metrics.loc[0, "roc_auc_perda"]),
    "taxa_perda_holdout": float(holdout_metrics.loc[0, "taxa_perda_holdout"]),
    "taxa_ganho_holdout": float(holdout_metrics.loc[0, "taxa_ganho_holdout"]),
    "qtd_features": len(selected_features),
    "qtd_dev": len(df_dev_model),
    "qtd_holdout": len(df_holdout_model),
    "ranking_risco_holdout_path": str(ranking_path),
}])

save_report(summary_step_3d, "82_3d_text_summary_final.csv")
summary_step_3d.T



## O que enviar para a próxima iteração

Envie prints ou os valores de:

1. `model_summary`
2. `holdout_metrics`
3. `holdout_topk_perda`
4. `holdout_topk_financeiro`
5. `comparison_3a_3b_3c_3d`
6. `summary_step_3d.T`
7. `feature_importance.head(30)`, se existir

## Decisão esperada

Se o 3D superar o 3A em PR-AUC, Top 5% risco ou Top 10% financeiro, ele pode virar candidato campeão. Caso contrário, manteremos o 3A como campeão provisório e seguiremos para calibração Venn-Abers.
